In [1]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

class Decorate:
    def __init__(self, base_estimator=DecisionTreeClassifier(), ensemble_size=10, artificial_size=100, random_state=42):
        self.base_estimator = base_estimator
        self.ensemble_size = ensemble_size
        self.artificial_size = artificial_size
        self.ensemble = []
        self.random_state = np.random.RandomState(random_state)

    def fit(self, X, y):
        print("🎯 Starting DECORATE Ensemble Training")
        print("-" * 55)

        # Train initial model
        model = self._clone_estimator()
        model.fit(X, y)
        self.ensemble.append(model)
        print(f"Model 1 trained on original data ✅")

        for i in range(1, self.ensemble_size):
            # Generate artificial data
            X_artificial = self._generate_artificial_data(X)
            y_artificial = self._generate_opposite_labels(X_artificial)
            
            # Combine real + artificial data
            X_new = np.vstack((X, X_artificial))
            y_new = np.hstack((y, y_artificial))
            
            # Train new model
            new_model = self._clone_estimator()
            new_model.fit(X_new, y_new)
            
            # Check accuracy improvement
            if self._evaluate_ensemble(X, y, new_model):
                self.ensemble.append(new_model)
                print(f"Model {len(self.ensemble)} added ✅ — Ensemble grew to {len(self.ensemble)} models")
            else:
                print(f"Model {i+1} rejected ❌ — No improvement in ensemble accuracy")

        print("-" * 55)
        print(f"🎉 Training complete! Total models in ensemble: {len(self.ensemble)}")

    def predict(self, X):
        preds = np.array([model.predict(X) for model in self.ensemble])
        return np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=preds)

    def _evaluate_ensemble(self, X, y, new_model):
        current_acc = accuracy_score(y, self.predict(X))
        new_preds = np.array([m.predict(X) for m in self.ensemble + [new_model]])
        combined_pred = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=new_preds)
        new_acc = accuracy_score(y, combined_pred)

        print(f"→ Current accuracy: {current_acc:.3f}, After adding model: {new_acc:.3f}")
        return new_acc >= current_acc

    def _generate_artificial_data(self, X):
        mins, maxs = X.min(axis=0), X.max(axis=0)
        return self.random_state.uniform(mins, maxs, (self.artificial_size, X.shape[1]))

    def _generate_opposite_labels(self, X_artificial):
        preds = self.predict(X_artificial)
        return 1 - preds  # flip 0->1, 1->0

    def _clone_estimator(self):
        return DecisionTreeClassifier()

# ------------------ MAIN CODE ------------------

# Create dataset
X, y = make_classification(
    n_samples=300, n_features=5, n_informative=3, n_classes=2, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train Decorate ensemble
decorate = Decorate(ensemble_size=7)
decorate.fit(X_train, y_train)

# Test performance
y_pred = decorate.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print("\n📊 Final Ensemble Accuracy on Test Data:", round(acc, 3))
print("✅ Decorate algorithm successfully implemented!")


🎯 Starting DECORATE Ensemble Training
-------------------------------------------------------
Model 1 trained on original data ✅
→ Current accuracy: 1.000, After adding model: 1.000
Model 2 added ✅ — Ensemble grew to 2 models
→ Current accuracy: 1.000, After adding model: 1.000
Model 3 added ✅ — Ensemble grew to 3 models
→ Current accuracy: 1.000, After adding model: 1.000
Model 4 added ✅ — Ensemble grew to 4 models
→ Current accuracy: 1.000, After adding model: 1.000
Model 5 added ✅ — Ensemble grew to 5 models
→ Current accuracy: 1.000, After adding model: 1.000
Model 6 added ✅ — Ensemble grew to 6 models
→ Current accuracy: 1.000, After adding model: 1.000
Model 7 added ✅ — Ensemble grew to 7 models
-------------------------------------------------------
🎉 Training complete! Total models in ensemble: 7

📊 Final Ensemble Accuracy on Test Data: 0.878
✅ Decorate algorithm successfully implemented!
